In [13]:
import pandas as pd
import matplotlib.pyplot as plt

pf = pd.read_csv("../../data/processed/data_ordenado.csv")

print(pf['fecha'])



0        1981-01-01
1        1981-01-02
2        1981-01-03
3        1981-01-04
4        1981-01-05
            ...    
16066    2024-12-27
16067    2024-12-28
16068    2024-12-29
16069    2024-12-30
16070    2024-12-31
Name: fecha, Length: 16071, dtype: object


In [19]:
n = len(pf)
train_df = pf[0:int(n*0.8)]
val_df = pf[int(n*0.8):int(n*0.9)]
test_df = pf[int(n*0.9):]
print(len(train_df['fecha']))
print(len(val_df['fecha']))
print(len(test_df['fecha']))

12856
1607
1608


In [9]:
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

# Ajustamos el escalador SOLO con los datos de entrenamiento
train_scaled_x = scaler_x.fit_transform(train_df)
# Supongamos que queremos predecir 'ciudad_bolivar' (columna 2)
train_scaled_y = scaler_y.fit_transform(train_df[['ciudad_bolivar']])

# Aplicamos la misma transformación a Validación y Prueba (sin re-ajustar)
val_scaled_x = scaler_x.transform(val_df)
val_scaled_y = scaler_y.transform(val_df[['ciudad_bolivar']])

test_scaled_x = scaler_x.transform(test_df)
test_scaled_y = scaler_y.transform(test_df[['ciudad_bolivar']])

NameError: name 'MinMaxScaler' is not defined

In [ ]:
def create_sequences(data_x, data_y, window_size=60):
    X, y = [], []
    for i in range(window_size, len(data_x)):
        X.append(data_x[i-window_size:i])
        y.append(data_y[i])
    return np.array(X), np.array(y)

window = 60
X_train, y_train = create_sequences(train_scaled_x, train_scaled_y, window)
X_val, y_val = create_sequences(val_scaled_x, val_scaled_y, window)
X_test, y_test = create_sequences(test_scaled_x, test_scaled_y, window)

# Resultado: X_train tendrá forma [muestras, 60, 26]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

model = Sequential([
    LSTM(100, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# CONFIGURACIÓN DE LOS GUARDIANES DEL ENTRENAMIENTO
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
]

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

In [ ]:
#evaluar exito del modelo 

plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida del Modelo')
plt.legend()
plt.show()

In [ ]:
#La precisión real (Des-normalización):

predicciones = model.predict(X_test)
# Convertimos de escala [0,1] a metros reales
predicciones_reales = scaler_y.inverse_transform(predicciones)
y_test_real = scaler_y.inverse_transform(y_test)

error_promedio = np.mean(np.abs(predicciones_reales - y_test_real))
print(f"El modelo tiene un error promedio de: {error_promedio:.2f} metros")